In [ ]:
!pip install flask flask-cors pandas

In [ ]:
import pandas as pd

data = {
    "month":["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"],
    "inventory":[620,680,650,720,760,700,790,840,810,880,920,960],
    "demand":[480,520,560,550,610,640,620,680,710,750,780,830],
    "on_time_delivery":[91,92,90,93,94,95,94,96,95,96,97,98],
    "supplier_count":[112,114,116,118,120,121,122,123,124,125,126,126],
    "logistics_cost":[42000,43500,42800,44000,45200,44800,46000,47000,46500,48000,49200,50000]
}

df = pd.DataFrame(data)
df.to_csv("supply_chain_data.csv",index=False)

df

,month,inventory,demand,on_time_delivery,supplier_count,logistics_cost
0,Jan,620,480,91,112,42000
1,Feb,680,520,92,114,43500
2,Mar,650,560,90,116,42800
3,Apr,720,550,93,118,44000
4,May,760,610,94,120,45200
5,Jun,700,640,95,121,44800
6,Jul,790,620,94,122,46000
7,Aug,840,680,96,123,47000
8,Sep,810,710,95,124,46500
9,Oct,880,750,96,125,48000


In [ ]:
import pandas as pd

df = pd.read_csv("supply_chain_data.csv")

print("Rows:",len(df))
print("Columns:",list(df.columns))
print(df.head())

Rows: 12
Columns: ['month', 'inventory', 'demand', 'on_time_delivery', 'supplier_count', 'logistics_cost']
  month  inventory  demand  on_time_delivery  supplier_count  logistics_cost
0   Jan        620     480                91             112           42000
1   Feb        680     520                92             114           43500
2   Mar        650     560                90             116           42800
3   Apr        720     550                93             118           44000
4   May        760     610                94             120           45200


In [ ]:
print("Average Inventory:",round(df["inventory"].mean(),2))
print("Average Demand:",round(df["demand"].mean(),2))
print("Average On-Time Delivery:",round(df["on_time_delivery"].mean(),2))
print("Total Logistics Cost:",df["logistics_cost"].sum())

Average Inventory: 777.5
Average Demand: 644.17
Average On-Time Delivery: 94.25
Total Logistics Cost: 549000


In [ ]:
%%writefile app.py
from flask import Flask,jsonify
from flask_cors import CORS
import pandas as pd

app=Flask(__name__)
CORS(app)

df=pd.read_csv("supply_chain_data.csv")

@app.route("/")
def home():
    return jsonify({"status":"success","message":"SupplyIQ Backend is running"})

@app.route("/api/overview")
def overview():
    return jsonify({
        "inventory_value":int(df["inventory"].sum()),
        "on_time_delivery":round(df["on_time_delivery"].mean(),1),
        "active_suppliers":int(df["supplier_count"].iloc[-1]),
        "risk_score":round(100-df["on_time_delivery"].mean(),1)
    })

@app.route("/api/inventory")
def inventory():
    return jsonify({
        "months":df["month"].tolist(),
        "inventory":df["inventory"].tolist(),
        "demand":df["demand"].tolist()
    })

@app.route("/api/suppliers")
def suppliers():
    average=round(df["on_time_delivery"].mean(),1)
    return jsonify({
        "on_time":average,
        "delayed":round(100-average,1)
    })

@app.route("/api/logistics")
def logistics():
    return jsonify({
        "months":df["month"].tolist(),
        "cost":df["logistics_cost"].tolist()
    })

@app.route("/api/forecast")
def forecast():
    recent=df["demand"].tail(4)
    growth=round(((recent.iloc[-1]-recent.iloc[0])/recent.iloc[0])*100,1)
    return jsonify({
        "recent_demand":recent.tolist(),
        "growth":growth,
        "next_month_forecast":round(recent.mean()*(1+growth/100),0)
    })

@app.route("/api/risks")
def risks():
    results=[]
    inventory=df["inventory"].iloc[-1]
    demand=df["demand"].iloc[-1]
    delivery=df["on_time_delivery"].mean()

    results.append({
        "type":"Stock Risk",
        "level":"High" if demand>inventory else "Low",
        "description":"Demand is higher than inventory." if demand>inventory else "Inventory is currently above demand."
    })

    results.append({
        "type":"Supplier Risk",
        "level":"High" if delivery<90 else "Low",
        "description":"Supplier delivery performance needs attention." if delivery<90 else "Supplier delivery performance is stable."
    })

    return jsonify(results)

@app.route("/api/ai-insights")
def ai_insights():
    return jsonify([
        f"Average monthly demand is {round(df['demand'].mean(),1)} units.",
        f"Average supplier on-time delivery is {round(df['on_time_delivery'].mean(),1)}%.",
        f"Latest demand is {df['demand'].iloc[-1]} units and should be monitored for inventory planning."
    ])

if __name__=="__main__":
    app.run(host="0.0.0.0",port=5000)

Writing app.py


In [ ]:
!python app.py

 * Serving Flask app 'app'
 * Debug mode: off
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
Press CTRL+C to quit
^C


In [ ]:
from app import app

client=app.test_client()

response=client.get("/api/overview")

print("Status:",response.status_code)
print("Data:",response.json)

Status: 200
Data: {'active_suppliers': 126, 'inventory_value': 9330, 'on_time_delivery': 94.2, 'risk_score': 5.8}


In [ ]:
from app import app

client=app.test_client()

apis=[
    "/",
    "/api/overview",
    "/api/inventory",
    "/api/suppliers",
    "/api/logistics",
    "/api/forecast",
    "/api/risks",
    "/api/ai-insights"
]

for api in apis:
    response=client.get(api)
    print(api,"→",response.status_code)

/ → 200
/api/overview → 200
/api/inventory → 200
/api/suppliers → 200
/api/logistics → 200
/api/forecast → 200
/api/risks → 200
/api/ai-insights → 200


In [ ]:
%%writefile requirements.txt
Flask
flask-cors
pandas
gunicorn

Writing requirements.txt


In [ ]:
import os

print(os.listdir())

['.config', 'app.py', 'supply_chain_data.csv', '__pycache__', 'requirements.txt', 'sample_data']
